# 14 · E-MNIST: espectro denso completo (n/N hasta 1) frente a N = 112 800 (Krylov)

Complementa al notebook 13. Allí N = 112 800 y solo existen los k = 1000 primeros modos (Krylov), así que el eje n/N llega a ~1e-2. Aquí se calculan **todos** los N autovalores del mismo operador DMAP congelado sobre submuestras de E-MNIST (split `balanced`) de **N = 4800, 9600 y 19 200** anclas, y se superponen los tamaños en el eje n/N, como los trabajos sobre la ley de Weyl que comparan tamaños de red, junto con la curva de Krylov de N = 112 800 (si la salida del notebook 13 está adjunta).

**Qué calcula.** Para cada tamaño y cada β = factor × β_base (misma heurística de la mediana sobre las entradas estandarizadas que `experiment.py`), `spectrum.py` con `method: dense` y `k: "all"` diagonaliza la matriz simétrica N × N con LAPACK y devuelve los N autovalores. Después dibuja λ_n = −4β ln μ_n contra n/N en log-log (una figura por factor con una curva por N, y una figura `spectrum_full` con todos los factores del N mayor), y el ajuste de Weyl λ_i ~ i^(2/d) en la ventana `weyl_window`.

**Por qué float64 en CPU.** La cola del espectro (μ_n muy pequeños) en float32 es ruido de redondeo, no espectro. El backend `torch` en CPU trabaja en float64 y LAPACK diagonaliza en float64, así que la cola es fiable hasta ~1e-16. Por eso este notebook no usa FlashDiffusion ni la GPU (el workflow lo lanza como sesión de CPU, sin acelerador, así que no gasta cuota de GPU). La memoria es una matriz N × N en float64 (≈ 3 GB a N = 19 200) más una copia transitoria al simetrizar.

**Piso de ruido.** Los modos con μ ≤ 1e-12 (constante `NOISE` de `spectrum.py`) no se dibujan ni se ajustan: el redondeo del diagonalizador es ~1e-16 relativo y se mantiene 4 décadas por debajo. Con kernels anchos (β pequeño) los autovalores caen por debajo de ese piso antes de llegar a n/N = 1, y la curva simplemente termina antes: es lo honesto. n/N llega a ~1 solo en los tamaños densos, nunca en la curva de Krylov (k = 1000, n/N ≲ 1e-2).

**Advertencia física.** Para una nube de puntos sobre una variedad fija, el espectro a n bajo depende de n (el volumen y la geometría de la variedad), no de n/N: los tamaños no tienen por qué colapsar a n/N pequeño. Pueden hacerlo en la parte de n alto, dominada por el muestreo (N finito). Compara por tanto las curvas con esa lectura, no como una prueba de colapso.

**Coste aproximado (estimación, sin medir en Kaggle).** La diagonalización domina y crece como N³: en un portátil de 10 hilos, 14 s a N = 4800, es decir ~15 min a N = 19 200 por β; en la CPU de Kaggle (4 núcleos) puede ser varias veces más. Hay 8 factores por tamaño; usa menos factores en `config_spectrum_emnist_full.json` si hace falta recortar.

**Datos:** adjunta el Kaggle Dataset **crawford/emnist** (el workflow de GitHub lo adjunta solo). Para la curva de N = 112 800 adjunta además la salida del kernel del notebook 13 (`dndr-13-emnist-kaggle`; el workflow lo hace); si falta, el notebook sigue sin esa curva.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
NB13 = 'dndr-13-emnist-kaggle'   # kernel whose output holds the Krylov run; its copy of the code is not a package source
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json', 'spectrum.py',
        'config_emnist.json', 'config_spectrum_emnist.json', 'config_spectrum_emnist_full.json']

# Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
sources = sorted({q.parent for q in INPUT.rglob('launch.py')
                  if (q.parent / 'spectral.py').exists() and NB13 not in q.parts})
archives = sorted(a for a in INPUT.rglob('dmap_vae_kaggle*.zip') if NB13 not in a.parts)
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
print('CPU float64, hilos:', torch.get_num_threads())
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
from IPython.display import display, Image

## Datos E-MNIST

Se prepara la caché **una sola vez** (la escritura es atómica).

In [ ]:
from data import ensure_emnist
EMNIST = ensure_emnist('/kaggle/working/emnist_balanced.npz', 'balanced')
import numpy as np
with np.load(EMNIST) as a:
    print({k: a[k].shape for k in a.files}, 'clases:', int(a['y_train'].max()) + 1)

## Espectro completo para N = 4800, 9600 y 19 200

Una ejecución de `spectrum.py` por tamaño (`k: "all"`, float64 en CPU, todas las imágenes de la submuestra como anclas). Cada tamaño escribe `spectrum.npz`, `spectrum_meta.json` y las tres figuras habituales en `spectrum_emnist_full/N<N>/`.

In [ ]:
full_cfg = json.loads((ROOT / 'config_spectrum_emnist_full.json').read_text())
full_cfg['data'] = str(EMNIST)
BASE = Path('/kaggle/working/spectrum_emnist_full')
SIZES = (4800, 9600, 19200)
dirs = []
for N in SIZES:
    cfg_n = dict(full_cfg, n=N, out=str(BASE / f'N{N}'))
    config = ROOT / f'active_spectrum_full_{N}.json'
    config.write_text(json.dumps(cfg_n, indent=2))
    started = time.perf_counter()
    subprocess.run([sys.executable, str(ROOT / 'spectrum.py'), str(config)], check=True)
    print(f'N = {N}: {(time.perf_counter() - started) / 60:.1f} min', flush=True)
    dirs.append(cfg_n['out'])

## Tamaños superpuestos y comparación con N = 112 800

`spectrum_sizes_f<factor>`: una curva por N contra n/N para cada β (cada una con su propio β en λ_n = −4β ln μ_n); la recta punteada es el ajuste de Weyl del N denso mayor. `spectrum_full`: todos los factores del N mayor. La curva de Krylov (N = 112 800, k = 1000) se añade si la salida del notebook 13 está adjunta; no tiene ajuste punteado.

In [ ]:
kry = next((p.parent for p in sorted(INPUT.rglob('spectrum_emnist/spectrum.npz'))
            if json.loads((p.parent / 'spectrum_meta.json').read_text()).get('method') == 'krylov'), None)
print('Krylov N = 112800:', kry or 'no encontrado, se omite esa curva (adjunta la salida del notebook 13)')
subprocess.run([sys.executable, str(ROOT / 'spectrum.py'), '--sizes', str(BASE), *dirs,
                *(['--krylov', str(kry)] if kry else [])], check=True)
display(Image(filename=str(BASE / 'spectrum_full.png')))
for f in sorted(BASE.glob('spectrum_sizes_f*.png'), key=lambda p: float(p.stem.split('_f')[1])):
    display(Image(filename=str(f)))

In [ ]:
metas = [json.loads((Path(d) / 'spectrum_meta.json').read_text()) for d in dirs]
ref = ({r['factor']: r['weyl_d'] for r in json.loads((kry / 'spectrum_meta.json').read_text())['weyl']}
       if kry else {})
print('Dimensión de Weyl d (ventana de la configuración)')
print(f"{'factor':>8}" + ''.join(f"{'N = ' + str(m['n_anchors']):>12}" for m in metas) + f"{'Krylov':>12}")
for j, r in enumerate(metas[0]['weyl']):
    f = r['factor']
    print(f"{f:>8g}" + ''.join(f"{m['weyl'][j]['weyl_d']:>12.2f}" for m in metas)
          + (f"{ref[f]:>12.2f}" if f in ref else f"{'-':>12}"))

## Archivos guardados

`spectrum_emnist_full/`: `N4800/`, `N9600/`, `N19200/` (cada uno con `spectrum.npz`, `spectrum_meta.json`, `spectrum_weyl`, `spectrum_heatmap` y `spectrum_local_d`), más `spectrum_full` y `spectrum_sizes_f<factor>` (.png/.pdf).